# 対話システムの基礎（ルールベース・検索型・タスク指向対話）

書籍 第4章「言語生成問題の解き方」（4-3 対話：チャットのやり取りをする）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、対話システムの基本設計である以下のアプローチを体験します。

1. **ルールベース（テンプレート / ELIZA 風パターンマッチ）**（4-3-1）
2. **用例検索型（TF-IDF 類似発話検索）**（4-3-2）
3. **タスク指向対話（意図分類・スロット抽出・対話状態管理）**: 複数ターンにわたって不足情報を聞き返し、条件変更にも追従する予約対話

## 1. 準備

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

---

## 2. ルールベース対話システム（4-3-1）

書籍 4-3-1 で紹介されている **ELIZA**（Weizenbaum, 1966）のように、正規表現で入力文のキーワードやパターンを捉え、あらかじめ用意したテンプレートに当てはめて応答を返すボットを作ります。
ここでは「研究室の先輩チャットボット（丁寧・相談役キャラ）」を想定してみます。

In [2]:
rules = [
    (r"(こんにちは|こんばんは|はじめまして|おはよう)",
     lambda m: "こんにちは！自然言語処理の勉強や研究のことで何か相談はありますか？"),
    (r"(.+)(が|は)(難しい|わからない|理解できない)",
     lambda m: f"「{m.group(1)}」は最初は難しく感じますよね。どのあたりで詰まっていますか？"),
    (r"(BERT|Transformer|GPT|LLM)",
     lambda m: f"{m.group(1)} は強力ですが、まずは TF-IDF やルールベースでベースラインを作るのもおすすめですよ。"),
    (r"(データ|アノテーション|コーパス)",
     lambda m: "データの質はモデル以上に大事です！仕様書を作って複数人で一致率を測ってみましたか？"),
    (r"(ありがとう|感謝|助かり)",
     lambda m: "どういたしまして！またいつでも声をかけてくださいね。"),
]

def rule_based_reply(user_text):
    for pattern, template_fn in rules:
        m = re.search(pattern, user_text)
        if m:
            return template_fn(m)
    return "なるほど、もう少し詳しく教えていただけますか？"

test_inputs = [
    "こんにちは！",
    "Transformerが難しいです",
    "アノテーションの進め方に迷っています",
    "今日はいい天気ですね",  # ルールに一致しない発話（フォールバック）
    "ありがとうございます！",
]

for u in test_inputs:
    print(f"ユーザ: {u}")
    print(f"ボット: {rule_based_reply(u)}\n")

ユーザ: こんにちは！
ボット: こんにちは！自然言語処理の勉強や研究のことで何か相談はありますか？

ユーザ: Transformerが難しいです
ボット: 「Transformer」は最初は難しく感じますよね。どのあたりで詰まっていますか？

ユーザ: アノテーションの進め方に迷っています
ボット: データの質はモデル以上に大事です！仕様書を作って複数人で一致率を測ってみましたか？

ユーザ: 今日はいい天気ですね
ボット: なるほど、もう少し詳しく教えていただけますか？

ユーザ: ありがとうございます！
ボット: どういたしまして！またいつでも声をかけてくださいね。



ルールベースは**想定したシナリオ内では破綻せずキャラクターの一貫性も保てる**一方、少し言い回しが変わったり想定外の発話が来たりすると画一的なフォールバック応答しか返せません。

---

## 3. 用例検索型（Retrieval-based）対話システム（4-3-2）

次に、書籍 4-3-2 の用例検索ベースとして、過去の Q&A・対話ペアのデータベースから「ユーザの入力に最も意味（表層）が近い質問」を TF-IDF コサイン類似度で検索し、対応する応答を返すシステムを作ります。
実運用では「類似度が一定の閾値未満なら応答せずフォールバックする」設計が重要です。

In [3]:
faq_pairs = [
    ("GPUがなくて学習が重いです", "まずは CPU で動く TF-IDF や線形モデルでベースラインを作り、その後 Google Colab の無料 T4 GPU を使うのがおすすめです。"),
    ("評価指標は何を使えばよいですか", "分類なら正解率に加えて F1、生成タスクなら BLEU や ROUGE、品質推定なら相関係数や Kendall's tau などをタスクに合わせて選びます。"),
    ("教師データがなかなか集まりません", "少量のデータにアノテーションして一致率を確認したり、ルールベースや事前学習モデルのゼロショットから始める手があります。"),
    ("形態素解析ツールはどれがおすすめですか", "Python から手軽に使うなら SudachiPy や fugashi（MeCab）、Web テキストの文分割なら Bunkai などがよく使われます。"),
    ("要約モデルを作る前に何をすべきですか", "ニュース記事などでは先頭数文を抜き出す「リード法」が強力なベースラインになるので、まずリード法と ROUGE で比較しましょう。"),
    ("翻訳モデルの精度を測るにはどうしますか", "参照訳がある場合は sacrebleu で BLEU を測り、参照訳がない場合は COMET-QE などの品質推定モデルが使えます。"),
]

questions = [q for q, _ in faq_pairs]
answers = [a for _, a in faq_pairs]

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3))
q_mat = vec.fit_transform(questions)

def retrieval_reply(user_text, threshold=0.25):
    u_vec = vec.transform([user_text])
    sims = cosine_similarity(u_vec, q_mat).ravel()
    best_idx = int(np.argmax(sims))
    best_sim = float(sims[best_idx])
    if best_sim < threshold:
        return best_sim, None, "すみません、その質問に合う回答が見つかりませんでした。別の言い方で質問してみてください。"
    return best_sim, questions[best_idx], answers[best_idx]

queries = [
    "Google ColabでGPUが使えないときはどうしたらいい？",
    "要約のベースライン手法を教えてください",
    "形態素解析のおすすめライブラリは？",
    "明日の東京の天気を教えてください",  # ドメイン外の質問（閾値で弾く）
]

for q in queries:
    sim, matched_q, ans = retrieval_reply(q)
    print(f"ユーザ: {q}")
    print(f"  [類似度={sim:.3f}, ヒット='{matched_q}']")
    print(f"ボット: {ans}\n")

ユーザ: Google ColabでGPUが使えないときはどうしたらいい？
  [類似度=0.417, ヒット='GPUがなくて学習が重いです']
ボット: まずは CPU で動く TF-IDF や線形モデルでベースラインを作り、その後 Google Colab の無料 T4 GPU を使うのがおすすめです。

ユーザ: 要約のベースライン手法を教えてください
  [類似度=0.278, ヒット='要約モデルを作る前に何をすべきですか']
ボット: ニュース記事などでは先頭数文を抜き出す「リード法」が強力なベースラインになるので、まずリード法と ROUGE で比較しましょう。

ユーザ: 形態素解析のおすすめライブラリは？
  [類似度=0.676, ヒット='形態素解析ツールはどれがおすすめですか']
ボット: Python から手軽に使うなら SudachiPy や fugashi（MeCab）、Web テキストの文分割なら Bunkai などがよく使われます。

ユーザ: 明日の東京の天気を教えてください
  [類似度=0.173, ヒット='None']
ボット: すみません、その質問に合う回答が見つかりませんでした。別の言い方で質問してみてください。



---

## 4. タスク指向対話（意図分類・スロット抽出・対話状態管理）（4-3）

雑談や FAQ の 1 問 1 答と異なり、ホテルやレストランの予約などの**タスク指向対話**では、必要な条件（**スロット**）がすべて揃うまで複数ターンにわたり**対話状態（Dialogue State）**を更新しながら不足項目を質問し続けます。

ここでは宿泊予約ボットを例に、次の 3 要素を組み合わせた最小構成を実装します。
1. **意図分類（Intent Classification）**: TF-IDF + ロジスティック回帰で「予約 / 条件変更 / キャンセル」を判定
2. **スロット抽出（Slot Filling）**: 発話中から `エリア`・`日付`・`人数` を抽出
3. **対話状態管理とポリシー（Dialogue State Tracking）**: 状態辞書に埋まったスロットを上書き保持し、未入力の必須スロットを順に聞き返す

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# 1) 意図分類器の学習（TF-IDF + ロジスティック回帰）
intent_train = [
    ("東京のホテルを予約したいです", "reserve"),
    ("来週金曜日に大阪で宿を取りたい", "reserve"),
    ("京都で2名で宿泊予約お願いします", "reserve"),
    ("明日1人で泊まれる部屋を予約できますか", "reserve"),
    ("やっぱり京都に変更できますか", "update"),
    ("人数を3名に変えてください", "update"),
    ("日付を来週土曜日に変更したいです", "update"),
    ("大人2名でお願いします", "provide_slot"),
    ("来週の金曜日にします", "provide_slot"),
    ("すみません、予約はキャンセルでお願いします", "cancel"),
    ("今回は取り消しにします", "cancel"),
]
X_int = [u for u, _ in intent_train]
y_int = [lbl for _, lbl in intent_train]
intent_clf = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3)), LogisticRegression(max_iter=1000, C=5.0))
intent_clf.fit(X_int, y_int)

# 2) スロット抽出器
AREAS = ["東京", "大阪", "京都", "札幌", "福岡"]
def extract_slots(text):
    found = {}
    for a in AREAS:
        if a in text:
            found["area"] = a
            break
    m_date = re.search(r"(今日|明日|明後日|今週末|来週(?:月|火|水|木|金|土|日)曜日|\d+月\d+日)", text)
    if m_date:
        found["date"] = m_date.group(1)
    m_ppl = re.search(r"(\d+\s*[名|人])", text)
    if m_ppl:
        found["people"] = m_ppl.group(1).replace(" ", "")
    return found

# 3) 対話状態管理（Dialogue State Manager）
class HotelBookingManager:
    REQUIRED_SLOTS = [("area", "ご希望のエリア（東京・大阪・京都など）"),
                      ("date", "ご宿泊日"),
                      ("people", "ご宿泊人数")]

    def __init__(self):
        self.reset()

    def reset(self):
        self.intent = None
        self.slots = {}
        self.status = "collecting"

    def step(self, user_text):
        pred_intent = intent_clf.predict([user_text])[0]
        extracted = extract_slots(user_text)

        if pred_intent == "cancel":
            self.reset()
            self.status = "cancelled"
            return pred_intent, extracted, "承知しました。予約手続きをキャンセルし、条件をリセットしました。"

        if pred_intent in ("reserve", "update") or self.intent is None:
            if self.intent != "reserve" or pred_intent == "reserve":
                self.intent = "reserve"
        self.slots.update(extracted)
        self.status = "collecting"

        missing = [(k, q) for k, q in self.REQUIRED_SLOTS if k not in self.slots]
        if missing:
            _, prompt = missing[0]
            return pred_intent, extracted, f"{prompt}を教えていただけますか？"
        self.status = "ready_to_book"
        s = self.slots
        return pred_intent, extracted, f"【条件確認】エリア: {s['area']} / 日付: {s['date']} / 人数: {s['people']} で空室を検索します。よろしければ確定してください。"

mgr = HotelBookingManager()
dialogue = [
    "来週金曜日に大阪で宿を予約したいです",
    "大人2名でお願いします",
    "すみません、やっぱり京都に変更できますか？",
    "ありがとうございます、今回の予約はキャンセルでお願いします",
]
logs = []
for turn, utt in enumerate(dialogue, 1):
    intent, ext, reply = mgr.step(utt)
    logs.append({
        "Turn": turn,
        "ユーザ発話": utt,
        "判定意図": intent,
        "今回抽出スロット": str(ext),
        "保持中の状態": str(dict(mgr.slots)),
        "ボット応答": reply,
    })
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(logs)

,Turn,ユーザ発話,判定意図,今回抽出スロット,保持中の状態,ボット応答
0,1,来週金曜日に大阪で宿を予約したいです,reserve,"{'area': '大阪', 'date': '来週金曜日'}","{'area': '大阪', 'date': '来週金曜日'}",ご宿泊人数を教えていただけますか？
1,2,大人2名でお願いします,provide_slot,{'people': '2名'},"{'area': '大阪', 'date': '来週金曜日', 'people': '2名'}",【条件確認】エリア: 大阪 / 日付: 来週金曜日 / 人数: 2名 で空室を検索します。よろしければ確定してください。
2,3,すみません、やっぱり京都に変更できますか？,update,{'area': '京都'},"{'area': '京都', 'date': '来週金曜日', 'people': '2名'}",【条件確認】エリア: 京都 / 日付: 来週金曜日 / 人数: 2名 で空室を検索します。よろしければ確定してください。
3,4,ありがとうございます、今回の予約はキャンセルでお願いします,cancel,{},{},承知しました。予約手続きをキャンセルし、条件をリセットしました。


### まとめ（4-3）

- **ルールベース**: 想定範囲の会話やキャラクター維持に強く、誤った情報を生成するリスクがありません。
- **検索ベース**: 言い回しの揺らぎに一定程度対応でき、人手が書いた自然な応答を返せます（閾値によるフォールバック設計が重要）。
- **タスク指向（意図分類・スロット抽出・状態管理）**: 複数ターンの会話履歴を構造化状態（スロット）として保持するため、「やっぱり京都に変更」のような部分修正にも対応して確実にタスクを完了できます。
- **生成ベース（Seq2Seq / LLM）**: 未知の質問にも柔軟に応答を生成できますが、ハルシネーションや状態管理・安全性の制御および評価が課題になります。